In [8]:
!ls

20220408_DATOS_ABIERTOS_UNIDAD_LUMINOSA_.csv  luminaria_Madrid.ipynb


In [19]:
!pip install pyproj


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 2.8 MB/s  0:00:03m 2.8 MB/s eta 0:00:01

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [9]:
import pandas as pd
import random
import json
import folium
from pyproj import Transformer

# luminaria

This dataset has the following information about lights in Madrid:
- tipo, type of the light: LED, DESCARGA
- different columns that specify the address
- districto, barrio
- X_UTM, Y_UTM

For our study we only need an id, address, and GPS position (convert X_UTM and Y_UTM to lat, long).
Also, we need to add an affluence stating the amount of people detected around the light.

In [10]:
# Load the CSV file (semicolon-separated)
df_luminaria = pd.read_csv("20220408_DATOS_ABIERTOS_UNIDAD_LUMINOSA_.csv", sep=';', quotechar='"', encoding='UTF-8') # or 'cp1252'
df_luminaria.head(3)

,"tipo_bloqu,C,50","TIPOL,C,254","VIA_CLASE,C,254","VIA_PAR,C,254","VIA_NOMBRE,C,254","CLASE_APP,C,254","NUMERO,N,10,0","COD_NDP,N,10,0","DISTRITO,N,10,0","BARRIO,N,10,0","X_UTM,N,19,11","Y_UTM,N,19,11"
0,LBE002,DESCARGA,CALLE,DE LAS,ERAS,NUMERO,3,20040016,16,4,"445648,48120000000","4480670,67020000000"
1,LBE002,DESCARGA,CALLE,DE LAS,ERAS,NUMERO,10,31024246,16,4,"445610,64160000000","4480740,99560000000"
2,GLED001,LED,PLAZA,DE,MAR DEL PLATA,NUMERO,12,11119324,16,4,"445680,07500000000","4480679,16380000000"


In [11]:
# Fix comma-decimal formatting
df_luminaria["X_UTM,N,19,11"] = df_luminaria["X_UTM,N,19,11"].astype(str).str.replace(",", ".").astype(float)
df_luminaria["Y_UTM,N,19,11"] = df_luminaria["Y_UTM,N,19,11"].astype(str).str.replace(",", ".").astype(float)


# Transformer for Madrid: ETRS89 / UTM 30N → WGS84
transformer = Transformer.from_crs("EPSG:25830", "EPSG:4326", always_xy=True)

# Vectorized transformation across entire columns
df_luminaria["longitude"], df_luminaria["latitude"] = transformer.transform(
    df_luminaria["X_UTM,N,19,11"].values,
    df_luminaria["Y_UTM,N,19,11"].values
)

df_luminaria = df_luminaria.drop(columns=["X_UTM,N,19,11", "Y_UTM,N,19,11"])

df_luminaria.head()

,"tipo_bloqu,C,50","TIPOL,C,254","VIA_CLASE,C,254","VIA_PAR,C,254","VIA_NOMBRE,C,254","CLASE_APP,C,254","NUMERO,N,10,0","COD_NDP,N,10,0","DISTRITO,N,10,0","BARRIO,N,10,0",longitude,latitude
0,LBE002,DESCARGA,CALLE,DE LAS,ERAS,NUMERO,3,20040016,16,4,-3.641197,40.474942
1,LBE002,DESCARGA,CALLE,DE LAS,ERAS,NUMERO,10,31024246,16,4,-3.641650,40.475573
2,GLED001,LED,PLAZA,DE,MAR DEL PLATA,NUMERO,12,11119324,16,4,-3.640825,40.475020
3,FFLED005,LED,CALLE,DEL,MAR AMARILLO,NUMERO,21,11119327,16,4,-3.640947,40.474709
4,FFLED005,LED,CALLE,DEL,MAR AMARILLO,NUMERO,19,11119326,16,4,-3.641163,40.474621


In [12]:
df_luminaria['address'] = df_luminaria['VIA_CLASE,C,254'] + " " + df_luminaria['VIA_PAR,C,254'] + " " + \
    df_luminaria['VIA_NOMBRE,C,254'] + " " + df_luminaria['NUMERO,N,10,0'].astype(str)

df_luminaria = df_luminaria.drop(columns=["VIA_CLASE,C,254", "VIA_PAR,C,254", "VIA_NOMBRE,C,254", "NUMERO,N,10,0"])


df_luminaria.head()

,"tipo_bloqu,C,50","TIPOL,C,254","CLASE_APP,C,254","COD_NDP,N,10,0","DISTRITO,N,10,0","BARRIO,N,10,0",longitude,latitude,address
0,LBE002,DESCARGA,NUMERO,20040016,16,4,-3.641197,40.474942,CALLE DE LAS ERAS 3
1,LBE002,DESCARGA,NUMERO,31024246,16,4,-3.641650,40.475573,CALLE DE LAS ERAS 10
2,GLED001,LED,NUMERO,11119324,16,4,-3.640825,40.475020,PLAZA DE MAR DEL PLATA 12
3,FFLED005,LED,NUMERO,11119327,16,4,-3.640947,40.474709,CALLE DEL MAR AMARILLO 21
4,FFLED005,LED,NUMERO,11119326,16,4,-3.641163,40.474621,CALLE DEL MAR AMARILLO 19


### Adding the affluence

Negative Binomial distribution

In [13]:
# parameters
np.random.seed(123)
mean_affluence = 1
dispersion = 0.3  # 0.3 = very variable, 1.0 = moderate variability


# Convert NB parameters to NumPy format
n = 1 / dispersion
p = 1 / (1 + mean_affluence * dispersion)

# Create the affluence column
df_luminaria["people_count"] = np.random.negative_binomial(
    n=n,
    p=p,
    size=len(df_luminaria)
)

df_luminaria.head(5)

,"tipo_bloqu,C,50","TIPOL,C,254","CLASE_APP,C,254","COD_NDP,N,10,0","DISTRITO,N,10,0","BARRIO,N,10,0",longitude,latitude,address,people_count
0,LBE002,DESCARGA,NUMERO,20040016,16,4,-3.641197,40.474942,CALLE DE LAS ERAS 3,0
1,LBE002,DESCARGA,NUMERO,31024246,16,4,-3.641650,40.475573,CALLE DE LAS ERAS 10,3
2,GLED001,LED,NUMERO,11119324,16,4,-3.640825,40.475020,PLAZA DE MAR DEL PLATA 12,0
3,FFLED005,LED,NUMERO,11119327,16,4,-3.640947,40.474709,CALLE DEL MAR AMARILLO 21,0
4,FFLED005,LED,NUMERO,11119326,16,4,-3.641163,40.474621,CALLE DEL MAR AMARILLO 19,0


In [14]:
total_people = df_luminaria["people_count"].sum()
print(total_people)

232873


In [19]:
df_luminaria = df_luminaria.drop(columns=['tipo_bloqu,C,50', 'TIPOL,C,254', 'CLASE_APP,C,254'])
df_luminaria

,"COD_NDP,N,10,0","DISTRITO,N,10,0","BARRIO,N,10,0",longitude,latitude,address,people_count
0,20040016,16,4,-3.641197,40.474942,CALLE DE LAS ERAS 3,0
1,31024246,16,4,-3.641650,40.475573,CALLE DE LAS ERAS 10,3
2,11119324,16,4,-3.640825,40.475020,PLAZA DE MAR DEL PLATA 12,0
3,11119327,16,4,-3.640947,40.474709,CALLE DEL MAR AMARILLO 21,0
4,11119326,16,4,-3.641163,40.474621,CALLE DEL MAR AMARILLO 19,0
...,...,...,...,...,...,...,...
233304,31022348,16,4,-3.641338,40.475097,CALLE DE LAS ERAS 4,0
233305,31022349,16,4,-3.641456,40.475300,CALLE DE LAS ERAS 6,0
233306,11119314,16,4,-3.641628,40.475450,CALLE DE LAS ERAS 9,0
233307,11119314,16,4,-3.641578,40.475474,CALLE DE LAS ERAS 9,1


In [21]:
df_luminaria = df_luminaria.rename(columns={
    'COD_NDP,N,10,0': 'code',
    'DISTRITO,N,10,0': 'district',
    'BARRIO,N,10,0': 'neighborhood',
    'longitude': 'longitude',
    'latitude': 'latitude',
    'address': 'address',
    'people_count': 'people_count'
})

In [24]:
np.random.seed(42)

# Define categories and their probabilities
states = ["on", "off", "broken"]
probabilities = [0.7, 0.2, 0.1]

# Add the new column
df_luminaria["status"] = np.random.choice(states, size=len(df_luminaria), p=probabilities)
df_luminaria.head()

,code,district,neighborhood,longitude,latitude,address,people_count,status
0,20040016,16,4,-3.641197,40.474942,CALLE DE LAS ERAS 3,0,on
1,31024246,16,4,-3.641650,40.475573,CALLE DE LAS ERAS 10,3,broken
2,11119324,16,4,-3.640825,40.475020,PLAZA DE MAR DEL PLATA 12,0,off
3,11119327,16,4,-3.640947,40.474709,CALLE DEL MAR AMARILLO 21,0,on
4,11119326,16,4,-3.641163,40.474621,CALLE DEL MAR AMARILLO 19,0,on


# Convertion of CSV into JSON -> NGSI-LD Entities

## Luminaries 

In [26]:

# Build NGSI-LD JSON entities
i=0
entities = []
for _, row in df_luminaria.iterrows():
    entity = {
        "id": f"urn:ngsi-ld:Luminaries:{i}",
        "type": "Luminaries",
        "district": {
            "type": "Property",
            "value": int(row["district"])
        },
        "neighborhood": {
            "type": "Property",
            "value": row["neighborhood"]
        },
        "status": {
            "type": "Property",
            "value": row["status"]
            
        },
        "people_count":  {
            "type": "Property",
            "value": int(row["people_count"])
        },
        "location": {
            "type": "GeoProperty",
            "value": {
                "type": "Point",
                "coordinates": [float(row["longitude"]), float(row["latitude"])]
            }
        }
    }
    i=i+1
    entities.append(entity)

# Convert to JSON array (pretty-printed)
json_output = json.dumps(entities, indent=2, ensure_ascii=False)
with open("luminaries.json", "w", encoding="utf-8") as f:
    f.write(json_output)
print("File written")

File written
